In [ ]:
import pandas as pd
import numpy as np
import re

from pathlib import Path
import os

# Local private CSV. Optionally set ECG_CSV_PATH in your environment.
CSV_PATH = Path(os.environ.get("ECG_CSV_PATH", "data/enrollment.csv"))

# What we WANT the features to be called (clean names)
FEATURES = ["beats", "mean_rr_ms", "std_rr_ms", "mean_peak", "std_peak", "energy"]

df = pd.read_csv(CSV_PATH)

print("Original columns:")
print(df.columns.tolist())

# ---- Normalize column names ----
# Examples:
# "|__beats" -> "beats"
# "|__mean_rr_ms" -> "mean_rr_ms"
# "|" -> "" (will be dropped later)
def normalize_col(c: str) -> str:
    c = c.strip()
    c = re.sub(r'^\|\__', '', c)   # remove leading "|__"
    c = re.sub(r'^\|', '', c)      # remove leading "|"
    c = c.strip("_")               # remove stray underscores
    return c

df.rename(columns={c: normalize_col(c) for c in df.columns}, inplace=True)

print("\nNormalized columns:")
print(df.columns.tolist())

# Drop empty column name if created (from "|" column)
if "" in df.columns:
    df = df.drop(columns=[""])

# ---- Check required columns exist now ----
missing = [c for c in FEATURES if c not in df.columns]
if missing:
    raise ValueError(
        f"Still missing required feature columns: {missing}\n"
        f"Columns found after normalize: {df.columns.tolist()}"
    )

# Keep ONLY biometric feature columns (drops lo_plus/lo_minus/t_ms/samples etc.)
df = df[FEATURES].copy()

# ---- Convert to numeric ----
for c in FEATURES:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# ---- Remove invalid placeholders ----
df.replace([-1, np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

# ---- Light sanity filtering ----
df = df[(df["beats"] >= 3) & (df["beats"] <= 45)]
df = df[(df["mean_rr_ms"] >= 250) & (df["mean_rr_ms"] <= 2500)]
df = df[(df["std_rr_ms"] >= 0) & (df["std_rr_ms"] <= 5000)]
df = df[(df["mean_peak"] >= 50) & (df["mean_peak"] <= 4095)]
df = df[(df["std_peak"] >= 0) & (df["std_peak"] <= 4095)]
df = df[(df["energy"] > 1)]

print("\nColumns used for ML:")
print(df.columns.tolist())
print("Clean rows:", len(df))

# ---- Build ENROLLED_USER template (mean + std) ----
if len(df) < 2:
    raise ValueError("Enrollment needs at least two valid rows after filtering. Collect more valid ECG windows; two rows are only a computational minimum, not proof of biometric accuracy.")

mu = df.mean()
sd = df.std(ddof=1).replace(0, 1e-6)
if not np.isfinite(mu.to_numpy()).all() or not np.isfinite(sd.to_numpy()).all() or not (sd > 0).all():
    raise ValueError("Enrollment produced invalid mean/standard-deviation values. Inspect your ECG export.")
Z_check = (df.to_numpy() - mu.to_numpy()) / sd.to_numpy()
dist_check = np.sqrt((Z_check ** 2).sum(axis=1))
suggested_threshold = float(dist_check.mean() + 3 * dist_check.std())
if not np.isfinite(suggested_threshold) or suggested_threshold <= 0:
    raise ValueError("Enrollment does not provide a finite, positive threshold. Collect varied valid windows and evaluate separately.")

print("\n=== VIRAJ TEMPLATE (paste into ESP32) ===")
for f in FEATURES:
    print(f"MU_{f} = {mu[f]:.6f}")

print()
for f in FEATURES:
    print(f"SD_{f} = {sd[f]:.6f}")

# ---- Suggest a threshold from training distances ----
X = df.values
Z = (X - mu.values) / sd.values
dist = np.sqrt((Z ** 2).sum(axis=1))

print("\nDistance stats (ENROLLED_USER):")
print("mean:", float(dist.mean()))
print("std :", float(dist.std()))
print("max :", float(dist.max()))
print("Suggested threshold (mean + 3*std):", suggested_threshold)
